# Deep Learning Model Selection for Heatwave Risk (Pakistan)

This notebook builds and compares deep learning models for multi-class `risk_label` prediction using temporal sequences from `df_model_forecast.csv`.

Models included:
- GRU + Attention
- LSTM + Attention
- TCN (Temporal Conv Net)
- Transformer Encoder

Selection target:
- Primary metric: **Validation Macro F1**
- Final evaluation: Test Macro F1, Weighted F1, per-class report, confusion matrix

In [ ]:
# If needed, install deep learning dependencies once.
# Uncomment and run if torch is not installed.
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

import os
import random
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Reproducibility and device setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Safer deterministic behavior when available.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

@dataclass
class Config:
    data_path: str = "../data/processed/df_model_forecast.csv"  # base table with target
    humidity_path: str = "../data/raw/pakistan_humidity_daily.csv"
    ndvi_path: str = "../data/raw/pakistan_ndvi_monthly.csv"
    merged_weather_scaled_path: str = "../data/processed/pakistan_weather_merged_scaled.csv"
    city_col: str = "city"
    date_col: str = "date"
    target_col: str = "risk_label"
    train_end_year: int = 2015
    val_end_year: int = 2019
    seq_len: int = 12
    batch_size: int = 32
    hidden_dim: int = 64
    embed_dim: int = 8
    num_layers: int = 2
    dropout: float = 0.25
    lr: float = 3e-4
    weight_decay: float = 1e-4
    epochs: int = 80
    patience: int = 12
    max_grad_norm: float = 1.0

CFG = Config()

In [ ]:
# Load and unify multi-source dataset (base + humidity + NDVI + optional merged weather file)
# 1) Drop all-NaN / all-zero numeric columns on each source before merge.
# 2) After merge: report missingness, drop merged columns that are still all-NaN or all-zero,
#    then train-median impute only columns with partial missingness (no leakage).


def _drop_uninformative_numeric_cols(frame: pd.DataFrame, key_cols: set) -> pd.DataFrame:
    """Drop numeric columns that are entirely NaN or entirely 0 (no variance). Keeps key_cols."""
    out = frame.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    drop_list = []
    for c in num_cols:
        if c in key_cols:
            continue
        s = pd.to_numeric(out[c], errors="coerce")
        if s.notna().sum() == 0:
            drop_list.append(c)
            continue
        if (s.fillna(0) == 0).all():
            drop_list.append(c)
    if drop_list:
        out = out.drop(columns=drop_list, errors="ignore")
        print(f"  Dropped uninformative numeric cols: {drop_list}")
    return out


# Base table (contains target and existing engineered features)
df = pd.read_csv(CFG.data_path)
if CFG.date_col in df.columns:
    df[CFG.date_col] = pd.to_datetime(df[CFG.date_col])
else:
    year_col = "year"
    month_col = "month"
    if year_col in df.columns and month_col in df.columns:
        df[CFG.date_col] = pd.to_datetime(dict(year=df[year_col], month=df[month_col], day=1))
    else:
        raise ValueError("No date/year-month columns found in base dataset.")

required_cols = {CFG.city_col, CFG.target_col, CFG.date_col}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns in base dataset: {missing_required}")

# Harmonize city name type early for safer joins.
df[CFG.city_col] = df[CFG.city_col].astype(str).str.strip()

# Year column for train-only imputation after merges
if "year" not in df.columns:
    df["year"] = df[CFG.date_col].dt.year
if "month" not in df.columns:
    df["month"] = df[CFG.date_col].dt.month
train_mask_early = df["year"] <= CFG.train_end_year

# --- Humidity daily -> monthly aggregation ---
if os.path.exists(CFG.humidity_path):
    hum = pd.read_csv(CFG.humidity_path)
    hum["time"] = pd.to_datetime(hum["time"], errors="coerce")
    hum = hum.dropna(subset=["time", "city"])
    hum["city"] = hum["city"].astype(str).str.strip()
    hum["year"] = hum["time"].dt.year
    hum["month"] = hum["time"].dt.month

    hum_cols = [c for c in ["rh_avg", "rh_max", "rh_min", "prcp", "et0"] if c in hum.columns]
    hum = _drop_uninformative_numeric_cols(hum, key_cols=set())
    hum_cols = [c for c in hum_cols if c in hum.columns]
    if hum_cols:
        hum_agg = hum.groupby(["city", "year", "month"], as_index=False)[hum_cols].mean()
        hum_agg = _drop_uninformative_numeric_cols(hum_agg, key_cols={"city", "year", "month"})
        hum_agg = hum_agg.rename(columns={c: f"hum_{c}" for c in hum_cols if c in hum_agg.columns})

        df = df.merge(
            hum_agg,
            how="left",
            left_on=[CFG.city_col, "year", "month"],
            right_on=["city", "year", "month"],
        )
        if "city" in df.columns and CFG.city_col != "city":
            df = df.drop(columns=["city"])

# --- NDVI monthly ---
if os.path.exists(CFG.ndvi_path):
    ndvi = pd.read_csv(CFG.ndvi_path)
    ndvi["time"] = pd.to_datetime(ndvi["time"], errors="coerce")
    ndvi = ndvi.dropna(subset=["time", "city"])
    ndvi["city"] = ndvi["city"].astype(str).str.strip()
    ndvi["year"] = ndvi["time"].dt.year
    ndvi["month"] = ndvi["time"].dt.month

    ndvi = _drop_uninformative_numeric_cols(ndvi, key_cols=set())
    if "ndvi" in ndvi.columns:
        ndvi_m = ndvi.groupby(["city", "year", "month"], as_index=False)["ndvi"].mean()
        ndvi_m = _drop_uninformative_numeric_cols(ndvi_m, key_cols={"city", "year", "month"})
        if "ndvi" in ndvi_m.columns:
            ndvi_m = ndvi_m.rename(columns={"ndvi": "ndvi_monthly"})
            df = df.merge(
                ndvi_m,
                how="left",
                left_on=[CFG.city_col, "year", "month"],
                right_on=["city", "year", "month"],
            )
            if "city" in df.columns and CFG.city_col != "city":
                df = df.drop(columns=["city"])

# --- Optional processed merged weather file daily -> monthly ---
if os.path.exists(CFG.merged_weather_scaled_path):
    wm = pd.read_csv(CFG.merged_weather_scaled_path)
    if {"time", "city"}.issubset(set(wm.columns)):
        wm["time"] = pd.to_datetime(wm["time"], errors="coerce")
        wm = wm.dropna(subset=["time", "city"])
        wm["city"] = wm["city"].astype(str).str.strip()
        wm["year"] = wm["time"].dt.year
        wm["month"] = wm["time"].dt.month

        wm = _drop_uninformative_numeric_cols(wm, key_cols=set())
        wm_candidate_cols = [
            c for c in ["rh_avg_scaled", "prcp_scaled", "et0_scaled", "rh_avg", "prcp", "et0"] if c in wm.columns
        ]
        if wm_candidate_cols:
            wm_m = wm.groupby(["city", "year", "month"], as_index=False)[wm_candidate_cols].mean()
            wm_m = _drop_uninformative_numeric_cols(wm_m, key_cols={"city", "year", "month"})
            wm_m = wm_m.rename(columns={c: f"wm_{c}" for c in wm_candidate_cols if c in wm_m.columns})
            df = df.merge(
                wm_m,
                how="left",
                left_on=[CFG.city_col, "year", "month"],
                right_on=["city", "year", "month"],
            )
            if "city" in df.columns and CFG.city_col != "city":
                df = df.drop(columns=["city"])

# --- Post-merge: missingness report, drop all-NaN / all-zero feature cols, partial train-median impute ---
exclude_meta = {
    CFG.target_col,
    CFG.city_col,
    CFG.date_col,
    "year",
    "month",
    "risk_lag_1", "risk_lag_3", "risk_lag_6",
}
numeric_after = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_after if c not in exclude_meta]

na_counts = df[candidate_features].isna().sum()
na_cols = na_counts[na_counts > 0]
print("\n--- Missing values after merge (column -> count) ---")
if len(na_cols) == 0:
    print("None.")
else:
    print(na_cols.sort_values(ascending=False).to_string())

# Drop merged numeric columns that are still entirely NaN or entirely 0
drop_after = []
for c in candidate_features:
    s = pd.to_numeric(df[c], errors="coerce")
    if s.notna().sum() == 0:
        drop_after.append(c)
    elif (s.fillna(0) == 0).all():
        drop_after.append(c)
if drop_after:
    df = df.drop(columns=drop_after, errors="ignore")
    print(f"\nDropped post-merge uninformative cols: {drop_after}")

# Recompute candidate feature names after drops
numeric_after = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_after if c not in exclude_meta]

# Partial missingness: train-median impute (same rule as next cell; applied here for merge-time gaps only)
train_med = df.loc[train_mask_early, candidate_features].median(numeric_only=True)
for c in candidate_features:
    if df[c].isna().any() and not df[c].isna().all():
        m = train_med.get(c, np.nan)
        if pd.isna(m):
            m = 0.0
        df[c] = df[c].fillna(m)

# Final missing check on numeric feature candidates
numeric_after = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_after if c not in exclude_meta]
still_na = df[candidate_features].isna().sum()
still_na = still_na[still_na > 0]
print("\n--- Missing values after train-median partial impute ---")
if len(still_na) == 0:
    print("None (or only non-numeric columns remain NA).")
else:
    print(still_na.to_string())
    # Remaining NA (e.g. all-NaN column not dropped): fill 0 for stability
    for c in still_na.index.tolist():
        df[c] = df[c].fillna(0.0)

# Keep only numeric feature columns and drop leakage-prone / metadata columns.
exclude_cols = {
    CFG.target_col,
    CFG.city_col,
    CFG.date_col,
    "year",
    "month",
    "risk_lag_1", "risk_lag_3", "risk_lag_6",
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

print(f"\nRows: {len(df):,}")
print(f"Cities: {df[CFG.city_col].nunique()}")
print(f"Classes: {sorted(df[CFG.target_col].dropna().unique().tolist())}")
print(f"Feature count after multi-source merge: {len(feature_cols)}")
print("Example added features:", [c for c in feature_cols if c.startswith(("hum_", "ndvi_", "wm_"))][:20])
print(feature_cols[:25])

In [ ]:
# Build city mappings, time split, and robust scaling/imputation

df = df.sort_values([CFG.city_col, CFG.date_col]).reset_index(drop=True)

# Ensure target is numeric and valid.
df[CFG.target_col] = pd.to_numeric(df[CFG.target_col], errors="coerce")
df = df.dropna(subset=[CFG.target_col]).copy()
df[CFG.target_col] = df[CFG.target_col].astype(int)

df["year_tmp"] = df[CFG.date_col].dt.year
train_mask = df["year_tmp"] <= CFG.train_end_year
val_mask = (df["year_tmp"] > CFG.train_end_year) & (df["year_tmp"] <= CFG.val_end_year)
test_mask = df["year_tmp"] > CFG.val_end_year

city_to_idx = {c: i for i, c in enumerate(sorted(df[CFG.city_col].unique()))}
df["city_idx"] = df[CFG.city_col].map(city_to_idx)
num_cities = len(city_to_idx)
num_classes = int(df[CFG.target_col].nunique())

# Convert features to numeric, remove inf, and impute with train medians only.
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)

train_feature_medians = df.loc[train_mask, feature_cols].median(numeric_only=True)
df[feature_cols] = df[feature_cols].fillna(train_feature_medians)
# If a column is fully missing in train, median can be NaN; force safe fallback.
df[feature_cols] = df[feature_cols].fillna(0.0)

scaler = StandardScaler()
scaler.fit(df.loc[train_mask, feature_cols])
df[feature_cols] = scaler.transform(df[feature_cols])
df[feature_cols] = np.nan_to_num(df[feature_cols].to_numpy(), nan=0.0, posinf=0.0, neginf=0.0)

print("Train rows:", int(train_mask.sum()))
print("Val rows:", int(val_mask.sum()))
print("Test rows:", int(test_mask.sum()))
print("Num cities:", num_cities, "Num classes:", num_classes)
print("Any non-finite in features:", ~np.isfinite(df[feature_cols].to_numpy()).all())

In [ ]:
# Dataset utilities
class SequenceDataset(Dataset):
    def __init__(self, X_seq, city_seq, y_seq):
        self.X = torch.tensor(X_seq, dtype=torch.float32)
        self.city = torch.tensor(city_seq, dtype=torch.long)
        self.y = torch.tensor(y_seq, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.city[idx], self.y[idx]


def make_sequences(df_part: pd.DataFrame, feature_cols: List[str], seq_len: int):
    X_all, city_all, y_all = [], [], []

    for _, g in df_part.groupby(CFG.city_col):
        g = g.sort_values(CFG.date_col)
        X = g[feature_cols].values
        y = g[CFG.target_col].values.astype(int)
        city_idx = g["city_idx"].values.astype(int)

        if len(g) < seq_len:
            continue

        for t in range(seq_len - 1, len(g)):
            X_all.append(X[t - seq_len + 1:t + 1])
            city_all.append(city_idx[t])
            y_all.append(y[t])

    return np.array(X_all), np.array(city_all), np.array(y_all)


def build_loaders(df: pd.DataFrame, feature_cols: List[str], seq_len: int, batch_size: int):
    train_df = df[df["year_tmp"] <= CFG.train_end_year].copy()
    val_df = df[(df["year_tmp"] > CFG.train_end_year) & (df["year_tmp"] <= CFG.val_end_year)].copy()
    test_df = df[df["year_tmp"] > CFG.val_end_year].copy()

    X_train, c_train, y_train = make_sequences(train_df, feature_cols, seq_len)
    X_val, c_val, y_val = make_sequences(val_df, feature_cols, seq_len)
    X_test, c_test, y_test = make_sequences(test_df, feature_cols, seq_len)

    train_ds = SequenceDataset(X_train, c_train, y_train)
    val_ds = SequenceDataset(X_val, c_val, y_val)
    test_ds = SequenceDataset(X_test, c_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, y_train, y_val, y_test

In [ ]:
# Model architectures
class AttentionPool(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (B, T, H)
        w = torch.softmax(self.score(x).squeeze(-1), dim=1)  # (B, T)
        context = torch.sum(x * w.unsqueeze(-1), dim=1)      # (B, H)
        return context


class RNNAttentionClassifier(nn.Module):
    def __init__(self, input_dim, num_cities, num_classes, rnn_type="gru", hidden_dim=64, num_layers=2, dropout=0.25, embed_dim=8):
        super().__init__()
        self.city_emb = nn.Embedding(num_cities, embed_dim)

        rnn_cls = nn.GRU if rnn_type.lower() == "gru" else nn.LSTM
        self.rnn = rnn_cls(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.attn = AttentionPool(hidden_dim * 2)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2 + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, city_idx):
        out, _ = self.rnn(x)
        ctx = self.attn(out)
        city_vec = self.city_emb(city_idx)
        logits = self.head(torch.cat([ctx, city_vec], dim=1))
        return logits


class TCNClassifier(nn.Module):
    def __init__(self, input_dim, num_cities, num_classes, hidden_dim=64, dropout=0.25, embed_dim=8):
        super().__init__()
        self.city_emb = nn.Embedding(num_cities, embed_dim)
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=2, dilation=1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=4, dilation=2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, city_idx):
        # x: (B, T, F) -> (B, F, T)
        x = x.transpose(1, 2)
        feat = self.net(x)
        pooled = feat.mean(dim=2)
        city_vec = self.city_emb(city_idx)
        logits = self.head(torch.cat([pooled, city_vec], dim=1))
        return logits


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_cities, num_classes, hidden_dim=64, num_layers=2, nhead=4, dropout=0.25, embed_dim=8):
        super().__init__()
        self.city_emb = nn.Embedding(num_cities, embed_dim)
        self.proj = nn.Linear(input_dim, hidden_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            dim_feedforward=hidden_dim * 2,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, city_idx):
        z = self.proj(x)
        z = self.encoder(z)
        pooled = z.mean(dim=1)
        city_vec = self.city_emb(city_idx)
        logits = self.head(torch.cat([pooled, city_vec], dim=1))
        return logits

In [ ]:
# Training and evaluation helpers

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    losses = []
    all_preds = []
    all_true = []
    skipped_batches = 0

    for xb, cb, yb in loader:
        xb, cb, yb = xb.to(device), cb.to(device), yb.to(device)

        # Hard guard against malformed batch values.
        if not torch.isfinite(xb).all():
            skipped_batches += 1
            continue

        with torch.set_grad_enabled(train):
            logits = model(xb, cb)
            if not torch.isfinite(logits).all():
                skipped_batches += 1
                continue

            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                skipped_batches += 1
                continue

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(yb.detach().cpu().numpy())

    if len(all_true) == 0:
        return np.nan, 0.0, 0.0, np.array([]), np.array([]), skipped_batches

    macro_f1 = f1_score(all_true, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_true, all_preds, average="weighted", zero_division=0)
    avg_loss = float(np.mean(losses)) if len(losses) > 0 else np.nan
    return avg_loss, macro_f1, weighted_f1, np.array(all_true), np.array(all_preds), skipped_batches


def train_model(model_name, model, train_loader, val_loader, class_weights):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

    best_state = None
    best_val_f1 = -1
    wait = 0
    history = []

    for epoch in range(1, CFG.epochs + 1):
        tr_loss, tr_macro, tr_w, _, _, tr_skipped = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_macro, va_w, _, _, va_skipped = run_epoch(model, val_loader, criterion, optimizer=None)

        if np.isfinite(va_macro):
            scheduler.step(va_macro)

        history.append({
            "epoch": epoch,
            "train_loss": tr_loss,
            "val_loss": va_loss,
            "train_macro_f1": tr_macro,
            "val_macro_f1": va_macro,
            "train_skipped_batches": tr_skipped,
            "val_skipped_batches": va_skipped,
        })

        if np.isfinite(va_macro) and va_macro > best_val_f1:
            best_val_f1 = va_macro
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"[{model_name}] Epoch {epoch:03d} | train_macro={tr_macro:.4f} | "
                f"val_macro={va_macro:.4f} | skipped(train/val)={tr_skipped}/{va_skipped}"
            )

        if wait >= CFG.patience:
            print(f"[{model_name}] Early stopped at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_f1

In [ ]:
# Build loaders and class weights
train_loader, val_loader, test_loader, y_train, y_val, y_test = build_loaders(
    df=df,
    feature_cols=feature_cols,
    seq_len=CFG.seq_len,
    batch_size=CFG.batch_size,
)

if len(y_train) == 0 or len(y_val) == 0 or len(y_test) == 0:
    raise ValueError("One of the sequence splits is empty. Reduce seq_len or check date splits.")

counts = np.bincount(y_train, minlength=num_classes)
weights = counts.sum() / np.maximum(counts, 1)
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32)

print("Train sequences:", len(y_train))
print("Val sequences:", len(y_val))
print("Test sequences:", len(y_test))
print("Class counts (train):", counts)
print("Class weights:", class_weights.numpy())

In [ ]:
# Quick label distribution diagnostics
for split_name, y_split in [("train", y_train), ("val", y_val), ("test", y_test)]:
    uniq, cnt = np.unique(y_split, return_counts=True)
    dist = {int(k): int(v) for k, v in zip(uniq, cnt)}
    print(f"{split_name} label distribution:", dist)

# If a split does not contain all classes, macro F1 can be harsh/unstable.
missing_in_train = set(range(num_classes)) - set(np.unique(y_train).tolist())
if missing_in_train:
    print("Warning: classes missing in train split:", missing_in_train)

In [ ]:
# Train and compare candidate models
input_dim = len(feature_cols)

candidates = {
    "GRU_Attn": RNNAttentionClassifier(
        input_dim=input_dim,
        num_cities=num_cities,
        num_classes=num_classes,
        rnn_type="gru",
        hidden_dim=CFG.hidden_dim,
        num_layers=CFG.num_layers,
        dropout=CFG.dropout,
        embed_dim=CFG.embed_dim,
    ),
    "LSTM_Attn": RNNAttentionClassifier(
        input_dim=input_dim,
        num_cities=num_cities,
        num_classes=num_classes,
        rnn_type="lstm",
        hidden_dim=CFG.hidden_dim,
        num_layers=CFG.num_layers,
        dropout=CFG.dropout,
        embed_dim=CFG.embed_dim,
    ),
    "TCN": TCNClassifier(
        input_dim=input_dim,
        num_cities=num_cities,
        num_classes=num_classes,
        hidden_dim=CFG.hidden_dim,
        dropout=CFG.dropout,
        embed_dim=CFG.embed_dim,
    ),
    "Transformer": TransformerClassifier(
        input_dim=input_dim,
        num_cities=num_cities,
        num_classes=num_classes,
        hidden_dim=CFG.hidden_dim,
        num_layers=2,
        nhead=4,
        dropout=CFG.dropout,
        embed_dim=CFG.embed_dim,
    ),
}

results = []
histories = {}
trained_models = {}

for name, model in candidates.items():
    print(f"\n=== Training {name} ===")
    best_model, hist_df, best_val_f1 = train_model(name, model, train_loader, val_loader, class_weights)
    trained_models[name] = best_model
    histories[name] = hist_df
    results.append({"model": name, "best_val_macro_f1": best_val_f1})

results_df = pd.DataFrame(results).sort_values("best_val_macro_f1", ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Plot validation curves
plt.figure(figsize=(10, 5))
for name, hist in histories.items():
    plt.plot(hist["epoch"], hist["val_macro_f1"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation Macro F1")
plt.title("Validation Macro F1 by Model")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Final test evaluation on the best validation model
best_name = results_df.iloc[0]["model"]
best_model = trained_models[best_name]

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.05)
test_loss, test_macro, test_weighted, y_true, y_pred, test_skipped = run_epoch(
    best_model,
    test_loader,
    criterion,
    optimizer=None,
)

print("Best model:", best_name)
print(f"Test loss: {test_loss:.4f}" if np.isfinite(test_loss) else "Test loss: NaN (check skipped batches)")
print(f"Test macro F1: {test_macro:.4f}")
print(f"Test weighted F1: {test_weighted:.4f}")
print("Skipped test batches:", test_skipped)

if len(y_true) == 0:
    raise RuntimeError("No valid test predictions were produced. Check feature quality and NaN handling.")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix - {best_name}")
plt.show()

In [ ]:
# City-wise comparison for the selected best model

def make_sequences_with_city_name(df_part: pd.DataFrame, feature_cols: List[str], seq_len: int):
    X_all, city_idx_all, city_name_all, y_all = [], [], [], []

    for city_name, g in df_part.groupby(CFG.city_col):
        g = g.sort_values(CFG.date_col)
        X = g[feature_cols].values
        y = g[CFG.target_col].values.astype(int)
        city_idx = g["city_idx"].values.astype(int)

        if len(g) < seq_len:
            continue

        for t in range(seq_len - 1, len(g)):
            X_all.append(X[t - seq_len + 1:t + 1])
            city_idx_all.append(city_idx[t])
            city_name_all.append(city_name)
            y_all.append(y[t])

    return np.array(X_all), np.array(city_idx_all), np.array(city_name_all), np.array(y_all)


def predict_arrays(model, X_np, city_idx_np, batch_size=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb = torch.tensor(X_np[i:i+batch_size], dtype=torch.float32, device=device)
            cb = torch.tensor(city_idx_np[i:i+batch_size], dtype=torch.long, device=device)
            logits = model(xb, cb)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            preds.extend(pred)
    return np.array(preds)


test_df_city = df[df["year_tmp"] > CFG.val_end_year].copy()
Xc, Cc, city_names, yc = make_sequences_with_city_name(test_df_city, feature_cols, CFG.seq_len)

if len(yc) == 0:
    raise RuntimeError("No city-level test sequences found. Try reducing seq_len.")

yp = predict_arrays(best_model, Xc, Cc)

rows = []
for city in sorted(np.unique(city_names)):
    mask = city_names == city
    y_true_city = yc[mask]
    y_pred_city = yp[mask]

    rows.append({
        "city": city,
        "samples": int(mask.sum()),
        "macro_f1": float(f1_score(y_true_city, y_pred_city, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true_city, y_pred_city, average="weighted", zero_division=0)),
        "accuracy": float((y_true_city == y_pred_city).mean()),
        "true_class_dist": str({int(k): int(v) for k, v in zip(*np.unique(y_true_city, return_counts=True))}),
        "pred_class_dist": str({int(k): int(v) for k, v in zip(*np.unique(y_pred_city, return_counts=True))}),
    })

city_results_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(f"City-wise performance of best model: {best_name}")
display(city_results_df)

# Plot city-wise macro and weighted F1
plot_df = city_results_df.sort_values("macro_f1", ascending=True)
plt.figure(figsize=(10, max(4, len(plot_df) * 0.6)))
plt.barh(plot_df["city"], plot_df["macro_f1"], alpha=0.8, label="Macro F1")
plt.barh(plot_df["city"], plot_df["weighted_f1"], alpha=0.5, label="Weighted F1")
plt.xlabel("Score")
plt.title(f"City-wise F1 Comparison ({best_name})")
plt.xlim(0, 1)
plt.legend()
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
# SHAP feature importance for the trained best model
# Note: This uses a sampled KernelExplainer to keep runtime manageable.

import shap

# Build test sequences again (sequence-level features)
test_df_shap = df[df["year_tmp"] > CFG.val_end_year].copy()
X_test_seq, C_test_seq, y_test_seq = make_sequences(test_df_shap, feature_cols, CFG.seq_len)

if len(X_test_seq) == 0:
    raise RuntimeError("No test sequences found for SHAP.")

# Flatten sequence features: (seq_len, n_features) -> seq_len*n_features, and append city_idx as last column.
X_test_flat = X_test_seq.reshape(len(X_test_seq), -1)
X_test_with_city = np.hstack([X_test_flat, C_test_seq.reshape(-1, 1)])

# Sample for speed
rng = np.random.default_rng(SEED)
background_n = min(100, len(X_test_with_city))
explain_n = min(120, len(X_test_with_city))

bg_idx = rng.choice(len(X_test_with_city), size=background_n, replace=False)
ex_idx = rng.choice(len(X_test_with_city), size=explain_n, replace=False)

background = X_test_with_city[bg_idx]
X_explain = X_test_with_city[ex_idx]

n_feat = len(feature_cols)

# Must match `best_model` device (avoids: input on CUDA, weights on CPU).
_shap_model_dev = next(best_model.parameters()).device

# Prediction wrapper expected by SHAP (returns class probabilities)
def predict_proba_from_flat(x_flat_with_city):
    x_flat_with_city = np.asarray(x_flat_with_city, dtype=np.float32)

    x_flat = x_flat_with_city[:, :-1]
    city_idx = np.rint(x_flat_with_city[:, -1]).astype(int)
    city_idx = np.clip(city_idx, 0, num_cities - 1)

    x_seq = x_flat.reshape(-1, CFG.seq_len, n_feat)

    xb = torch.tensor(x_seq, dtype=torch.float32, device=_shap_model_dev)
    cb = torch.tensor(city_idx, dtype=torch.long, device=_shap_model_dev)

    best_model.eval()
    with torch.no_grad():
        logits = best_model(xb, cb)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

# Explain probabilities for all classes
explainer = shap.KernelExplainer(predict_proba_from_flat, background)
shap_values = explainer.shap_values(X_explain, nsamples=120)

# Build readable feature names for flattened sequence inputs + city id
flat_feature_names = []
for t in range(CFG.seq_len):
    for f in feature_cols:
        flat_feature_names.append(f"t-{CFG.seq_len-1-t}:{f}")
flat_feature_names.append("city_idx")

def _flat_to_base_feature_means(mean_flat, n_feat, seq_len):
    """Collapse [seq_len*n_feat + city] vector to per-base-feature mean |SHAP|."""
    temporal_part = mean_flat[:-1].reshape(seq_len, n_feat)
    return temporal_part.mean(axis=0)

# Average |SHAP| over explain samples, per class, then aggregate time -> one value per base feature
if isinstance(shap_values, list):
    sv = np.mean(np.stack([np.abs(v) for v in shap_values], axis=0), axis=0)
    n_cls = len(shap_values)
    per_class_columns = {}
    for c in range(n_cls):
        m_flat = np.abs(shap_values[c]).mean(axis=0)
        per_class_columns[f"mean_abs_shap_class_{c}"] = _flat_to_base_feature_means(
            m_flat, n_feat, CFG.seq_len
        )
else:
    sv_arr = np.abs(np.asarray(shap_values))
    if sv_arr.ndim == 3:
        sv = sv_arr.mean(axis=2)
    else:
        sv = sv_arr
    n_cls = None
    per_class_columns = {}

mean_abs = sv.mean(axis=0)
per_feature_importance = _flat_to_base_feature_means(mean_abs, n_feat, CFG.seq_len)

# Signed SHAP (mean over classes, then over explain rows) for symmetric importance plot
if isinstance(shap_values, list):
    signed_sv = np.mean(np.stack([np.asarray(v) for v in shap_values], axis=0), axis=0)
else:
    sv_s = np.asarray(shap_values)
    if sv_s.ndim == 3:
        signed_sv = sv_s.mean(axis=2)
    else:
        signed_sv = sv_s
mean_signed_flat = signed_sv.mean(axis=0)
per_feature_signed = _flat_to_base_feature_means(mean_signed_flat, n_feat, CFG.seq_len)

# Optional: spread of importance across explain samples (aggregated to base features)
std_per_sample = []
for i in range(sv.shape[0]):
    std_per_sample.append(_flat_to_base_feature_means(sv[i], n_feat, CFG.seq_len))
std_per_sample = np.stack(std_per_sample, axis=0)
feature_std_across_samples = std_per_sample.std(axis=0)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": per_feature_importance,
    "mean_shap_signed": per_feature_signed,
})
for k, col in per_class_columns.items():
    importance_df[k] = col
importance_df["std_across_explain_samples"] = feature_std_across_samples
importance_df = importance_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
importance_df.insert(0, "rank", np.arange(1, len(importance_df) + 1))

import matplotlib.ticker as mticker

pd.set_option("display.max_rows", 30)
print(f"Mean |SHAP| for {len(importance_df)} features (model: {best_name}) — see plot for all")
display(importance_df.head(30))

# All features, x-axis = log10 so small and large |SHAP| are visible (tick marks at 10^n)
plot_all = importance_df.sort_values("mean_abs_shap", ascending=True)
vals = np.maximum(plot_all["mean_abs_shap"].to_numpy(dtype=float), 1e-20)
_n = len(plot_all)
fig, ax = plt.subplots(figsize=(10, max(6, min(0.32 * _n, 32))))
ax.barh(plot_all["feature"], vals, color="coral")
ax.set_xscale("log", base=10)
ax.set_xlabel("Mean |SHAP| (log10: tick marks at powers of 10)")
ax.set_title(f"All {_n} features by mean |SHAP| — {best_name}")
ax.xaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=20))
ax.xaxis.set_major_formatter(mticker.LogFormatter(10, labelOnlyBase=False))
ax.grid(axis="x", which="both", alpha=0.3)
fig.tight_layout()
plt.show()

os.makedirs("../outputs/figures", exist_ok=True)
importance_df.to_csv("../outputs/figures/shap_all_features_numerical.csv", index=False)
importance_df.to_csv("../outputs/figures/shap_feature_importance_best_model.csv", index=False)
print("Saved full table to ../outputs/figures/shap_all_features_numerical.csv")

In [ ]:
# Save SHAP importance plots to disk (does NOT recompute SHAP — uses `importance_df` in memory,
# or loads from CSV if you restarted the kernel).

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHAP_CSV_FALLBACK = "../outputs/figures/shap_all_features_numerical.csv"
out_dir = "../outputs/figures"
os.makedirs(out_dir, exist_ok=True)

if "importance_df" not in globals():
    importance_df = None

if importance_df is None or len(importance_df) == 0:
    if os.path.exists(SHAP_CSV_FALLBACK):
        importance_df = pd.read_csv(SHAP_CSV_FALLBACK)
        print("Loaded importance_df from:", SHAP_CSV_FALLBACK)
    else:
        raise FileNotFoundError(
            "No `importance_df` in memory and CSV missing. "
            "Run the SHAP cell once (saves CSV) or set SHAP_CSV_FALLBACK path."
        )

import matplotlib.ticker as mticker

_model_label = globals().get("best_name", "model")
if "mean_abs_shap" in importance_df.columns:
    dfp = importance_df.sort_values("mean_abs_shap", ascending=True)
    vals = np.maximum(dfp["mean_abs_shap"].to_numpy(dtype=float), 1e-20)
    _n = len(dfp)
    fig, ax = plt.subplots(figsize=(10, max(6, min(0.32 * _n, 32))))
    ax.barh(dfp["feature"], vals, color="coral")
    ax.set_xscale("log", base=10)
    ax.set_xlabel("Mean |SHAP| (log10: tick marks at powers of 10)")
    ax.set_title(f"All {_n} features by mean |SHAP| — {_model_label}")
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=20))
    ax.xaxis.set_major_formatter(mticker.LogFormatter(10, labelOnlyBase=False))
    ax.grid(axis="x", which="both", alpha=0.3)
    fig.tight_layout()
    p = os.path.join(out_dir, f"shap_all_features_log10_{str(_model_label).lower()}.png")
    fig.savefig(p, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", p)
else:
    print("No mean_abs_shap column; cannot save figure.")

In [ ]:
# Save best model and metadata
os.makedirs("../models/deep_learning", exist_ok=True)

save_path = f"../models/deep_learning/{best_name.lower()}_best.pt"
torch.save({
    "model_name": best_name,
    "model_state_dict": best_model.state_dict(),
    "feature_cols": feature_cols,
    "city_to_idx": city_to_idx,
    "config": CFG.__dict__,
}, save_path)

results_path = "../models/deep_learning/dl_model_comparison.csv"
results_df.to_csv(results_path, index=False)

print("Saved best model to:", save_path)
print("Saved comparison table to:", results_path)

### SHAP for all candidate models (separate block)

Runs **Kernel SHAP** once per entry in `trained_models` (GRU, LSTM, TCN, Transformer).  
Each figure uses a **log₁₀ x-axis** with **left limit `10⁻¹⁰`** so small `mean |SHAP|` values stay visible.  
**Prerequisite:** run the training cell that fills `trained_models`. Does not modify the earlier single-model SHAP cell.

In [ ]:
# SHAP feature importance for EVERY model in `trained_models` (log10 x-axis from 1e-10)
# Does not change earlier SHAP cells — run after the training loop that builds `trained_models`.

import shap
import matplotlib.ticker as mticker

if "trained_models" not in globals() or not trained_models:
    raise RuntimeError("Run the model-training cell first so `trained_models` is non-empty.")

LOG_X_MIN = 1e-10
NSAMPLES = 120

def _flat_to_base_agg(mean_flat, n_feat, seq_len):
    temporal_part = mean_flat[:-1].reshape(seq_len, n_feat)
    return temporal_part.mean(axis=0)

# One shared background / explain set for all models (fair comparison)
test_df_m = df[df["year_tmp"] > CFG.val_end_year].copy()
X_seq, C_seq, _ = make_sequences(test_df_m, feature_cols, CFG.seq_len)
if len(X_seq) == 0:
    raise RuntimeError("No test sequences for SHAP.")
X_flat = X_seq.reshape(len(X_seq), -1)
X_with_city = np.hstack([X_flat, C_seq.reshape(-1, 1)])
rng = np.random.default_rng(SEED)
bg_n = min(100, len(X_with_city))
ex_n = min(120, len(X_with_city))
background = X_with_city[rng.choice(len(X_with_city), size=bg_n, replace=False)]
X_explain = X_with_city[rng.choice(len(X_with_city), size=ex_n, replace=False)]
n_feat = len(feature_cols)

def make_predict_proba(mdl):
    dev = next(mdl.parameters()).device

    def predict_proba_from_flat(x_flat_with_city):
        x = np.asarray(x_flat_with_city, dtype=np.float32)
        xf = x[:, :-1]
        city_idx = np.clip(np.rint(x[:, -1]).astype(int), 0, num_cities - 1)
        x_seq = xf.reshape(-1, CFG.seq_len, n_feat)
        xb = torch.tensor(x_seq, dtype=torch.float32, device=dev)
        cb = torch.tensor(city_idx, dtype=torch.long, device=dev)
        mdl.eval()
        with torch.no_grad():
            logits = mdl(xb, cb)
            return torch.softmax(logits, dim=1).cpu().numpy()

    return predict_proba_from_flat

shap_importance_by_model = {}
os.makedirs("../outputs/figures", exist_ok=True)

for model_name, model in trained_models.items():
    print(f"\n--- SHAP: {model_name} ---")
    pred_fn = make_predict_proba(model)
    explainer = shap.KernelExplainer(pred_fn, background)
    sv = explainer.shap_values(X_explain, nsamples=NSAMPLES)

    if isinstance(sv, list):
        sv_abs = np.mean(np.stack([np.abs(v) for v in sv], axis=0), axis=0)
    else:
        arr = np.abs(np.asarray(sv))
        sv_abs = arr.mean(axis=2) if arr.ndim == 3 else arr

    mean_flat = sv_abs.mean(axis=0)
    per_feature = _flat_to_base_agg(mean_flat, n_feat, CFG.seq_len)
    idf = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": per_feature})
    idf = idf.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_importance_by_model[model_name] = idf

    csv_path = f"../outputs/figures/shap_{model_name.lower()}_mean_abs_all_features.csv"
    idf.to_csv(csv_path, index=False)
    print("Saved:", csv_path)

    plot_df = idf.sort_values("mean_abs_shap", ascending=True)
    vals = np.maximum(plot_df["mean_abs_shap"].to_numpy(dtype=float), LOG_X_MIN)
    nrows = len(plot_df)
    fig, ax = plt.subplots(figsize=(10, max(6, min(0.3 * nrows, 32))))
    ax.barh(plot_df["feature"], vals, color="coral")
    ax.set_xscale("log", base=10)
    ax.set_xlim(LOG_X_MIN, max(float(np.max(vals)) * 1.05, LOG_X_MIN * 100))
    ax.set_xlabel("Mean |SHAP| (log10 scale; left end = 10⁻¹⁰)")
    ax.set_title(f"All features — mean |SHAP| — {model_name}")
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=20))
    ax.xaxis.set_major_formatter(mticker.LogFormatter(10, labelOnlyBase=False))
    ax.grid(axis="x", which="both", alpha=0.3)
    fig.tight_layout()
    png_path = f"../outputs/figures/shap_all_models_log10_{model_name.lower()}.png"
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved figure:", png_path)

print("\nDone. Dict `shap_importance_by_model` holds DataFrames per model.")